In [ ]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-chroma
!pip install -q chromadb
!pip install -q sentence-transformers
!pip install -q pypdf
!pip install -q transformers
!pip install -q accelerate
!pip install -q langchain-text-splitters
!pip install -q openai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 128.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

**step 2**

In [ ]:
import os
import torch

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceBgeEmbeddings

from langchain_chroma import Chroma

from transformers import pipeline


print("All libraries imported successfully")

/tmp/ipykernel_25023/2148669326.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


All libraries imported successfully


**step 3**

In [ ]:
pdf_path = "cn book.pdf"

print("PDF Selected:", pdf_path)

PDF Selected: cn book.pdf


In [ ]:
loader = PyPDFLoader(pdf_path)

documents = loader.load()

print("Total Pages:", len(documents))

Total Pages: 1180


In [ ]:
print(documents[0].page_content[:1000])

In [ ]:
print(documents[0].metadata)

{'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2024-02-15T04:13:36+00:00', 'author': 'Ian Neil', 'moddate': '2025-01-02T11:02:20+05:30', 'subject': 'Master cybersecurity fundamentals and pass the SY0-701 exam on your first attempt', 'title': 'CompTIA Security+ SY0-701 Certification Guide, Third Edition (for DUC PHAM)', 'source': 'cn book.pdf', 'total_pages': 1180, 'page': 0, 'page_label': '1'}


**STEP 4: Advanced Text Chunking**

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=800,

    chunk_overlap=200,

    separators=[
        "\n\n",
        "\n",
        ".",
        " ",
        ""
    ]

)


print("Text splitter created")

Text splitter created


In [ ]:
chunks = text_splitter.split_documents(documents)


print("Total chunks created:", len(chunks))

Total chunks created: 2123


In [ ]:
print(chunks[0].page_content)

CompTIA Security+ SY0-701
Certification Guide
Third Edition
Copyright © 2024 Packt Publishing
All rights reserved. No part of this book may be reproduced,
stored in a retrieval system, or transmitted in any form or by
any means, without the prior written permission of the
publisher, except in the case of brief quotations embedded in
critical articles or reviews.
Every eﬀort has been made in the preparation of this book to
ensure the accuracy of the information presented. However, the
information contained in this book is sold without warranty,
either express or implied. Neither the author, nor Packt
Publishing or its dealers and distributors, will be held liable for


In [ ]:
print(chunks[0].metadata)

{'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2024-02-15T04:13:36+00:00', 'author': 'Ian Neil', 'moddate': '2025-01-02T11:02:20+05:30', 'subject': 'Master cybersecurity fundamentals and pass the SY0-701 exam on your first attempt', 'title': 'CompTIA Security+ SY0-701 Certification Guide, Third Edition (for DUC PHAM)', 'source': 'cn book.pdf', 'total_pages': 1180, 'page': 3, 'page_label': '4'}


**STEP 5: Create Embeddings + Store in ChromaDB**

In [ ]:
embedding_model = HuggingFaceBgeEmbeddings(

    model_name="BAAI/bge-large-en-v1.5",

    model_kwargs={
    "device":"cuda" if torch.cuda.is_available() else "cpu"
},

    encode_kwargs={
        "normalize_embeddings": True
    }

)


print("Embedding model loaded successfully")

/tmp/ipykernel_25023/2378769223.py:1: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceBgeEmbeddings(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Embedding model loaded successfully


In [ ]:
from langchain_chroma import Chroma


vector_db = Chroma.from_documents(

    documents=chunks,

    embedding=embedding_model,

    persist_directory="./chroma_db"

)


print("ChromaDB created successfully")

print(
    "Documents:",
    vector_db._collection.count()
)

ChromaDB created successfully
Documents: 2123


**STEP 6: Create Retriever + Test Semantic Search**

In [ ]:
retriever = vector_db.as_retriever(

    search_type="similarity",

    search_kwargs={
        "k":2
    }

)

print("Improved retriever created")

Improved retriever created


In [ ]:
print("Documents in vector database:")
print(vector_db._collection.count())

Documents in vector database:
2123


In [ ]:
queries = [

    "What is TCP protocol?",
    "Explain OSI model layers",
    "What is routing algorithm?",
    "Difference between TCP and UDP"

]


for query in queries:

    print("\n==========================")
    print("QUESTION:")
    print(query)


    results = retriever.invoke(query)


    if len(results) == 0:

        print("\nNo relevant document found")

        continue


    print("\nTOP RESULT:")

    print(results[0].page_content[:500])


    print("\nSOURCE:")

    print(results[0].metadata)


QUESTION:
What is TCP protocol?

TOP RESULT:
Hypertext
Transport
Protocol
(HTTP)
80 Web browser
Post Oﬃce
Protocol 3
110 Pulls mail from a
mail server; no
copy is left on the
mail server
Network Time
Protocol (NTP)
123 Time
synchronization
NETBIOS UDP 137–139 NETBIOS to IP
address resolution
Internet
Message Access
Protocol (IMAP
4)
143 Pulls mail from a
mail server
Simple
Network
Management
UDP 161 Notiﬁes the status
and creates

SOURCE:
{'creationdate': '2024-02-15T04:13:36+00:00', 'title': 'CompTIA Security+ SY0-701 Certification Guide, Third Edition (for DUC PHAM)', 'author': 'Ian Neil', 'page_label': '690', 'creator': 'PyPDF', 'page': 689, 'moddate': '2025-01-02T11:02:20+05:30', 'subject': 'Master cybersecurity fundamentals and pass the SY0-701 exam on your first attempt', 'producer': 'PyPDF', 'total_pages': 1180, 'source': 'cn book.pdf'}

QUESTION:
Explain OSI model layers

TOP RESULT:
Layer 3 switch: A Layer 3 switch, also known as a multilayer
switch, is a network switch that 

In [ ]:
from openai import OpenAI

print("OpenAI client imported successfully")

OpenAI client imported successfully


In [ ]:
import os

os.environ["OPENROUTER_API_KEY"] = ""

print("API key added")

API key added


In [ ]:
from openai import OpenAI
import os


client = OpenAI(

    base_url="https://openrouter.ai/api/v1",

    api_key=os.environ["OPENROUTER_API_KEY"]

)


print("OpenRouter connected successfully")

OpenRouter connected successfully


In [ ]:
docs = retriever.invoke(
    "What is TCP protocol?"
)


for i, doc in enumerate(docs):

    print("\n========== CHUNK", i+1, "==========")

    print(doc.page_content[:1000])

    print("\nSOURCE:")
    print(doc.metadata)


========== CHUNK 1 ==========
Hypertext
Transport
Protocol
(HTTP)
80 Web browser
Post Oﬃce
Protocol 3
110 Pulls mail from a
mail server; no
copy is left on the
mail server
Network Time
Protocol (NTP)
123 Time
synchronization
NETBIOS UDP 137–139 NETBIOS to IP
address resolution
Internet
Message Access
Protocol (IMAP
4)
143 Pulls mail from a
mail server
Simple
Network
Management
UDP 161 Notiﬁes the status
and creates

SOURCE:
{'creator': 'PyPDF', 'source': 'cn book.pdf', 'page_label': '690', 'total_pages': 1180, 'author': 'Ian Neil', 'creationdate': '2024-02-15T04:13:36+00:00', 'producer': 'PyPDF', 'moddate': '2025-01-02T11:02:20+05:30', 'subject': 'Master cybersecurity fundamentals and pass the SY0-701 exam on your first attempt', 'page': 689, 'title': 'CompTIA Security+ SY0-701 Certification Guide, Third Edition (for DUC PHAM)'}

========== CHUNK 2 ==========
orientated Transmission Control Port (TCP), which uses a
three-way handshake, and the connectionless User Datagram
Protocol (UD

In [ ]:
def openrouter_rag_answer(question):

    # Retrieve relevant chunks
    docs = retriever.invoke(question)
    print("Retrieved Context:")

    for doc in docs:
        print("----------------")
        print(doc.page_content[:500])


    # Check if no documents are retrieved
    if len(docs) == 0:

        return "Information not available in the document."


    # Prepare context with source pages
    context = ""

    for doc in docs:

        page = doc.metadata.get(
            "page",
            "unknown"
        )

        context += f"""

Source Page: {page}

{doc.page_content}

"""


    # Create improved prompt
    prompt = f"""

Rules:

1. Use ONLY the provided context.
2. Do NOT use your own knowledge.
3. Do NOT add examples that are not present in the context.
4. Do NOT mention general knowledge.
5. If the context is insufficient, answer:
   "Information not available in the document."

6. Provide a clear and complete answer.
7. Include important technical details from the context.
8. If the context describes a process, explain it step-by-step.
9. Use bullet points for better readability.
10. Do not remove important information from the context.

Context:

{context}


Question:

{question}


Answer:

"""


    # OpenRouter API call
    response = client.chat.completions.create(

        model="meta-llama/llama-3.1-8b-instruct",

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0

    )


    answer = response.choices[0].message.content


    return answer.strip()

In [ ]:
answer = openrouter_rag_answer(
    "What is TCP protocol?"
)


print(answer)

Retrieved Context:
----------------
Hypertext
Transport
Protocol
(HTTP)
80 Web browser
Post Oﬃce
Protocol 3
110 Pulls mail from a
mail server; no
copy is left on the
mail server
Network Time
Protocol (NTP)
123 Time
synchronization
NETBIOS UDP 137–139 NETBIOS to IP
address resolution
Internet
Message Access
Protocol (IMAP
4)
143 Pulls mail from a
mail server
Simple
Network
Management
UDP 161 Notiﬁes the status
and creates
----------------
orientated Transmission Control Port (TCP), which uses a
three-way handshake, and the connectionless User Datagram
Protocol (UDP). Both types are covered in detail in the
following subsections.
TCP
TCP is connection-orientated and uses what is known as the
three-way handshake to ensure the delivery of data. There are
three stages of this handshake – SYN, followed by SYN-ACK,
and ﬁnally, Acknowledgement (ACK), as depicted in Figure
18.2:
TCP is connection-orientated and uses a three-way handshake.


In [ ]:
answer = openrouter_rag_answer(
   "Explain TCP (Transmission Control Protocol) and its three-way handshake."
)


print(answer)

Retrieved Context:
----------------
orientated Transmission Control Port (TCP), which uses a
three-way handshake, and the connectionless User Datagram
Protocol (UDP). Both types are covered in detail in the
following subsections.
TCP
TCP is connection-orientated and uses what is known as the
three-way handshake to ensure the delivery of data. There are
three stages of this handshake – SYN, followed by SYN-ACK,
and ﬁnally, Acknowledgement (ACK), as depicted in Figure
18.2:
----------------
Figure 18.2: The TCP/IP three-way handshake
As you can see, Figure 18.2 shows the sequence of packet
exchanges that can create a session between A (the sender) and
B (the receiver), through which they will later exchange data.
This process is referred to as the three-way handshake and can
be broken down into the following three steps:
Here is the explanation of TCP and its three-way handshake based on the provided context:

**TCP (Transmission Control Protocol)**

* TCP is a connection-oriented protoc

In [ ]:
answer = openrouter_rag_answer(
  "Explain DHCP and how it is used for IP address allocation."
)


print(answer)

Retrieved Context:
----------------
common examples of which are Wireshark or the Linux-based
Tcpdump. A trace can be conducted by capturing packets, i.e.,
saving the data in a packet capture (PCAP) form for later
analysis. An example of packet capturing is troubleshooting
why a user did not receive an IP address from the DHCP server
that automates IP address allocation. Allocating an IP address
automatically is done by the exchange of four packets. If the
cybersecurity administrator only sees the ﬁrst packet but no
second packet,
----------------
Figure 10.3: Network layout
As you can see from the preceding ﬁgure, each computer has an
Ethernet cable that plugs into a wall jack, which is then
connected to the patch panel by cables that are laid under
ﬂoors or above ceilings. A cable runs from the patch panel to an
Ethernet port on the switch, as depicted in Figure 10.3. This
Ethernet cable is routed through a conduit to safeguard it from
damage. Since it’s quite simple to connect a cab